#### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

response = model.invoke("What is the tranformer architecture?")

response

AIMessage(content='## The Transformer Architecture – A High‑Level Overview\n\nThe **Transformer** is a neural‑network architecture introduced in the paper *“Attention Is All You Need”* (Vaswani\u202fet\u202fal., 2017). It was designed for sequence‑to‑sequence tasks (e.g., machine translation) and quickly became the foundation for almost all modern large‑language models (BERT, GPT, T5, etc.).  \n\nBelow is a concise, step‑by‑step description of the core ideas, the main building blocks, and why the design works so well.\n\n---\n\n### 1️⃣ Core Philosophy: “Attention‑only” Networks  \n\n* **Self‑attention** replaces recurrent or convolutional operations.  \n* Each token can directly attend to every other token in the same sequence, regardless of distance.  \n* This gives the model **global, content‑based connectivity** and makes it highly parallelizable (unlike RNNs).\n\n---\n\n### 2️⃣ High‑Level Structure  \n\n```\nInput Sequence  →  Embedding + Positional Encoding  →  Encoder Stack  →  (

In [3]:
from langchain.tools import tool

# create tool

@tool # this is called decorator
def get_weather(location: str) -> str:
    """
    Get the current weather for a given location.
    """
    # In a real implementation, you would call a weather API here.
    # For this example, we'll just return a dummy response.
    return f"The current weather in {location} is sunny with a temperature of 25°C."


# bind this tool with llm
model_with_tool = model.bind_tools([get_weather])

In [4]:
# calling the model with the tool

response = model_with_tool.invoke("What is the weather in New York?")

print(response)

content='' additional_kwargs={'reasoning_content': 'User asks: "What is the weather in New York?" We need to get current weather using function. Use get_weather with location "New York".', 'tool_calls': [{'id': 'fc_85fc5af1-1aff-4e3c-8520-7548ce47cb8c', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 130, 'total_tokens': 189, 'completion_time': 0.122187173, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.00514757, 'prompt_tokens_details': None, 'queue_time': 0.311155057, 'total_time': 0.127334743}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a07f96-16c2-70e0-b1ea-8ebb0e30c804-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_85fc5af1-1aff-4e3c-8520-7548ce47

In [5]:
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


Tool: get_weather
Args: {'location': 'New York'}


Tool Execution Loops

In [6]:
# step 1: Model generates tool calls
messages = [{"role": "user", "content": "what's the weather in Bostonn?"}]

ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass result back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)

The current weather in Boston is sunny with a temperature of 25 °C. Let me know if you’d like a forecast for the next few days or details for a different location!


In [7]:
messages

[{'role': 'user', 'content': "what's the weather in Bostonn?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what\'s the weather in Bostonn?" Likely typo "Boston". We need to fetch weather using get_weather function. Provide location "Boston". Use function.', 'tool_calls': [{'id': 'fc_0cc6a9d4-724b-4297-8ddc-f01bcc23c9ca', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 131, 'total_tokens': 196, 'completion_time': 0.141647097, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.005678378, 'prompt_tokens_details': None, 'queue_time': 0.353187822, 'total_time': 0.147325475}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07f96-4058-7ac1-ac57-2a6d7d62c34a-0', tool_cal